In [1]:
import pandas as pd

df = pd.read_csv("retail_sales_analysis_clean.csv")

print("Dataset shape:", df.shape)

Dataset shape: (62884, 35)


In [2]:
df.head()

,Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code,Product Name,...,Continent,Birthday,Store Country,Store State,Square Meters,Open Date,Revenue USD,Cost USD,Gross Profit USD,Year-Month
0,366000,1,2016-01-01,NaN,265598,10,1304,1,CAD,Contoso Lens Adapter M450 White,...,North America,3/23/1971,Canada,Nunavut,1210.0,2015-04-04,68.00,31.27,36.73,2016-01
1,366001,1,2016-01-01,2016-01-13,1269051,0,1048,2,USD,A. Datum SLR Camera X136 Silver,...,North America,11/20/1995,Online,Online,NaN,2010-01-01,854.00,282.94,571.06,2016-01
2,366001,2,2016-01-01,2016-01-13,1269051,0,2007,1,USD,Fabrikam Microwave 1.5CuFt X1100 Black,...,North America,11/20/1995,Online,Online,NaN,2010-01-01,665.94,220.64,445.30,2016-01
3,366002,1,2016-01-01,2016-01-12,266019,0,1106,7,CAD,Contoso SLR Camera M146 Orange,...,North America,8/24/1962,Online,Online,NaN,2010-01-01,2254.00,1036.56,1217.44,2016-01
4,366002,2,2016-01-01,2016-01-12,266019,0,373,1,CAD,Adventure Works Laptop8.9 E0890 White,...,North America,8/24/1962,Online,Online,NaN,2010-01-01,326.00,166.20,159.80,2016-01


In [3]:
#First, identify the revenue column:- Revenue USD, Cost USD, Gross profit USD

In [4]:
df.columns.tolist()

['Order Number',
 'Line Item',
 'Order Date',
 'Delivery Date',
 'CustomerKey',
 'StoreKey',
 'ProductKey',
 'Quantity',
 'Currency Code',
 'Product Name',
 'Brand',
 'Color',
 'Unit Cost USD',
 'Unit Price USD',
 'SubcategoryKey',
 'Subcategory',
 'CategoryKey',
 'Category',
 'Gender',
 'Name',
 'City',
 'State Code',
 'Customer State',
 'Zip Code',
 'Customer Country',
 'Continent',
 'Birthday',
 'Store Country',
 'Store State',
 'Square Meters',
 'Open Date',
 'Revenue USD',
 'Cost USD',
 'Gross Profit USD',
 'Year-Month']

In [5]:
#Total revenue generated by the retailer. Here we create a reusable python function so AI assistant can call it whenever someone asks about total revenue. 

In [6]:
def get_total_revenue(data):
    total_revenue = data["Revenue USD"].sum()
    return round(total_revenue, 2)

get_total_revenue(df)

55755479.59

In [7]:
#Gross Profit function: 

In [8]:
def get_total_gross_profit(data):
    total_profit = data["Gross Profit USD"].sum()
    return round(total_profit, 2)

get_total_gross_profit(df)

32662688.38

In [9]:
#gross_profit_margin

In [10]:
def get_gross_profit_margin(data):
    revenue = get_total_revenue(data)
    gross_profit = get_total_gross_profit(data)

    if revenue == 0:
        return 0

    margin = (gross_profit / revenue) * 100
    return round(margin, 2)

get_gross_profit_margin(df)

58.58

In [11]:
#total_orders

In [12]:
def get_total_orders(data):
    total_orders = data["Order Number"].nunique()
    return total_orders

get_total_orders(df)

26326

In [13]:
#average_order_value

In [14]:
def get_average_order_value(data):
    revenue = get_total_revenue(data)
    orders = get_total_orders(data)

    if orders == 0:
        return 0

    aov = revenue / orders
    return round(aov, 2)

get_average_order_value(df)

2117.89

In [15]:
#Revenue function:Before filtering by year, we need to verify its data type

In [16]:
print(df["Order Date"].dtype)
print(df["Order Date"].head())

object
0    2016-01-01
1    2016-01-01
2    2016-01-01
3    2016-01-01
4    2016-01-01
Name: Order Date, dtype: object


In [17]:
# Before AI assistant can filter revenue by year

In [100]:
# Convert Order Date from text to datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])

# Verify the conversion
print(df["Order Date"].dtype)
print(df["Order Date"].head())

datetime64[ns]
0   2016-01-01
1   2016-01-01
2   2016-01-01
3   2016-01-01
4   2016-01-01
Name: Order Date, dtype: datetime64[ns]


In [19]:
df["Order Date"].dt.year

0        2016
1        2016
2        2016
3        2016
4        2016
         ... 
62879    2021
62880    2021
62881    2021
62882    2021
62883    2021
Name: Order Date, Length: 62884, dtype: int32

In [20]:
#Dynamic revenue fun:total revenue in a specific year without changing the underlying code.

In [21]:
def get_revenue_by_year(data, year):
    filtered_data = data[data["Order Date"].dt.year == year]

    revenue = filtered_data["Revenue USD"].sum()

    return round(revenue, 2)

# Test the function for 2019
get_revenue_by_year(df, 2019)

18264382.48

In [22]:
get_revenue_by_year(df, 2019)

18264382.48

In [23]:
get_revenue_by_year(df, 2020)

9294632.14

In [24]:
#Filter revenue by Year AND Product Category

In [25]:
def get_revenue_by_year_category(data, year, category):
    filtered_data = data[
        (data["Order Date"].dt.year == year) &
        (data["Category"] == category)
    ]

    revenue = filtered_data["Revenue USD"].sum()

    return round(revenue, 2)

# Test: Computers revenue in 2019
get_revenue_by_year_category(df, 2019, "Computers")

6958430.89

In [26]:
#Sales Channel filtering

In [27]:
#We're combining three business filters: year, product category, and sales channel. 
#This will allow our future AI assistant to answer more specific questions without creating a separate function for every possible combination.

In [28]:
def get_revenue_by_filters(data, year, category, channel):
    filtered_data = data[
        (data["Order Date"].dt.year == year) &
        (data["Category"] == category)
    ]

    if channel == "Online":
        filtered_data = filtered_data[
            filtered_data["StoreKey"] == 0
        ]

    elif channel == "Physical Store":
        filtered_data = filtered_data[
            filtered_data["StoreKey"] != 0
        ]

    revenue = filtered_data["Revenue USD"].sum()

    return round(revenue, 2)


# Test: Online Computers revenue in 2019
get_revenue_by_filters(df, 2019, "Computers", "Online")

1500897.15

In [29]:
#Online vs. Physical Store revenue

In [30]:
online = get_revenue_by_filters(
    df, 2019, "Computers", "Online"
)

physical = get_revenue_by_filters(
    df, 2019, "Computers", "Physical Store"
)

total = get_revenue_by_year_category(
    df, 2019, "Computers"
)

print("Online Revenue:", online)
print("Physical Store Revenue:", physical)
print("Combined Revenue:", round(online + physical, 2))
print("Total Computers Revenue:", total)
print("Validation:", round(online + physical, 2) == total)

Online Revenue: 1500897.15
Physical Store Revenue: 5457533.74
Combined Revenue: 6958430.89
Total Computers Revenue: 6958430.89
Validation: True


In [31]:
#Make the filters optional

In [32]:
def calculate_revenue(data, year=None, category=None, channel=None):
    filtered_data = data.copy()

    if year is not None:
        filtered_data = filtered_data[
            filtered_data["Order Date"].dt.year == year
        ]

    if category is not None:
        filtered_data = filtered_data[
            filtered_data["Category"] == category
        ]

    if channel == "Online":
        filtered_data = filtered_data[
            filtered_data["StoreKey"] == 0
        ]

    elif channel == "Physical Store":
        filtered_data = filtered_data[
            filtered_data["StoreKey"] != 0
        ]

    return round(filtered_data["Revenue USD"].sum(), 2)


print("2019 Revenue:", calculate_revenue(df, year=2019))

#testing with different questions

print("Computers Revenue:", calculate_revenue(df, category="Computers"))

print("Online Revenue 2020:", calculate_revenue(df, year=2020, channel="Online"))

print("Online Computers 2019:", calculate_revenue(
    df, year=2019, category="Computers", channel="Online"
))

2019 Revenue: 18264382.48
Computers Revenue: 19301595.46
Online Revenue 2020: 2069950.9
Online Computers 2019: 1500897.15


In [33]:
#natural-language questions

In [34]:
#We'll start with a simple Python prototype before connecting an LLM.

In [35]:
import re

def extract_year(question):
    match = re.search(r"\b(20\d{2})\b", question)

    if match:
        return int(match.group(1))

    return None


# Test questions
print(extract_year("What was revenue in 2019?"))
print(extract_year("Show me sales for 2020"))
print(extract_year("What is total revenue?"))

2019
2020
None


In [36]:
#Business Question

In [37]:
def answer_revenue_question(question, data):
    year = extract_year(question)

    revenue = calculate_revenue(data, year=year)

    return f"Revenue: ${revenue:,.2f}"


answer_revenue_question("What was revenue in 2019?", df)

'Revenue: $18,264,382.48'

In [38]:
#What was Computers revenue in 2019?

In [39]:
def extract_category(question, data):
    categories = data["Category"].dropna().unique()

    for category in categories:
        if category.lower() in question.lower():
            return category

    return None


print(extract_category("What was Computers revenue in 2019?", df))
print(extract_category("Show me revenue for Cell phones", df))
print(extract_category("What was total revenue?", df))

Computers
Cell phones
None


In [40]:
#Connect category and year to revenue

In [41]:
def answer_revenue_question(question, data):
    year = extract_year(question)
    category = extract_category(question, data)

    revenue = calculate_revenue(
        data,
        year=year,
        category=category
    )

    return f"Revenue: ${revenue:,.2f}"

# Test
answer_revenue_question(
    "What was Computers revenue in 2019?", df
)

'Revenue: $6,958,430.89'

In [42]:
#What was Online Computers revenue in 2019?

In [43]:
def extract_channel(question):
    question = question.lower()

    if "online" in question:
        return "Online"

    elif "physical" in question or "in-store" in question:
        return "Physical Store"

    return None


print(extract_channel("Show online revenue in 2019"))
print(extract_channel("What were physical store sales?"))
print(extract_channel("What was total revenue?"))

Online
Physical Store
None


In [44]:
#Connect all three filters: What was Online Computers revenue in 2019?

In [45]:
def answer_revenue_question(question, data):
    year = extract_year(question)
    category = extract_category(question, data)
    channel = extract_channel(question)

    revenue = calculate_revenue(
        data,
        year=year,
        category=category,
        channel=channel
    )

    return f"Revenue: ${revenue:,.2f}"


# Test
answer_revenue_question(
    "What was Online Computers revenue in 2019?", df
)

'Revenue: $1,500,897.15'

In [46]:
#check with multiple questions

In [98]:
questions = [
    "What was total revenue in 2019?",
    "What was Computers revenue in 2019?",
    "What was Online revenue in 2020?",
    "What was Physical Store Computers revenue in 2019?",
    "What was total revenue?"
]

for question in questions:
    print("Question:", question)
    print("Answer:", answer_revenue_question(question, df))
    print()

Question: What was total revenue in 2019?
Answer: Revenue: $18,264,382.48

Question: What was Computers revenue in 2019?
Answer: Revenue: $6,958,430.89

Question: What was Online revenue in 2020?
Answer: Revenue: $2,069,950.90

Question: What was Physical Store Computers revenue in 2019?
Answer: Revenue: $5,457,533.74

Question: What was total revenue?
Answer: Revenue: $55,755,479.59



In [48]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [102]:
print("Kernel is working!")

Kernel is working!


In [104]:
import ollama

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": "What is revenue in business? Answer in one sentence."
        }
    ]
)

print(response["message"]["content"])

Revenue in business refers to the total amount of money earned by a company from its sales of goods or services over a specific period of time, typically measured in a calendar year or fiscal year.


In [ ]:
#Test Ollama's understanding of our business questions

In [106]:
question = "How much did we make from online computer sales in 2019?"

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": f"""
            Identify the metric, year, category, and sales channel
            from this retail business question.

            Question: {question}

            Return only these four fields.
            """
        }
    ]
)

print(response["message"]["content"])

Here are the requested fields:

- Metric: Sales
- Year: 2019
- Category: Online computer sales
- Sales Channel: Online


In [ ]:
#Make Ollama return structured data

In [108]:
import json

question = "How much did we make from online computer sales in 2019?"

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": f"""
            Extract the following fields from this retail question:
            metric, year, category, channel.

            Question: {question}

            Use these exact values:
            metric: revenue
            category: Computers
            channel: Online

            Return JSON only.
            """
        }
    ],
    format="json"
)

result = json.loads(response["message"]["content"])

print(result)

{'metric': 'revenue', 'year': 2019, 'category': 'Computers', 'channel': 'Online'}


In [110]:
print("Dataset loaded:", df.shape)

Dataset loaded: (62884, 35)


In [112]:
print("Revenue function available:", callable(calculate_revenue))

Revenue function available: True


In [114]:
#Connecting Ollama to our revenue calculation

In [116]:
year = int(result["year"])
category = result["category"]
channel = result["channel"]

revenue = calculate_revenue(
    df,
    year=year,
    category=category,
    channel=channel
)

print(f"Question: {question}")
print(f"Revenue: ${revenue:,.2f}")

Question: How much did we make from online computer sales in 2019?
Revenue: $1,500,897.15


In [118]:
question = "What was Computers revenue in 2019?"

revenue = calculate_revenue(
    df,
    year=2019,
    category="Computers"
)

print(f"Question: {question}")
print(f"Revenue: ${revenue:,.2f}")

Question: What was Computers revenue in 2019?
Revenue: $6,958,430.89


In [120]:
#Test Ollama with a new question

In [122]:
question = "What was Computers revenue in 2019?"

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": f"""
            Extract metric, year, category, and channel
            from this business question:

            {question}

            Return JSON only.
            Use null if a field is not mentioned.
            """
        }
    ],
    format="json"
)

result = json.loads(response["message"]["content"])

print(result)

{'metric': 'revenue', 'year': 2019, 'category': 'Computers', 'channel': None}


In [ ]:
#Connect AI interpretation to revenue calculation

In [125]:
revenue = calculate_revenue(
    df,
    year=result["year"],
    category=result["category"],
    channel=result["channel"]
)

print(f"Question: {question}")
print(f"Revenue: ${revenue:,.2f}")

Question: What was Computers revenue in 2019?
Revenue: $6,958,430.89


In [129]:
#the above code does Instead of manually entering 2019 or "Computers", Python takes those values directly from Ollama's JSON output.

In [ ]:
#reusable AI Assistant function

In [135]:
#Right now, we're running separate code cells to interpret and answer questions. Let's combine them into one simple function.

In [151]:
def ask_retail_ai(question):
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": f"""
                Extract metric, year, category, and channel
                from this retail business question:

                {question}

                Return JSON only.
                Use null for missing fields.
                Use "Computers" for the Computers category.
                Use "Online" or "Physical Store" for channels.
                """
            }
        ],
        format="json"
    )

    result = json.loads(response["message"]["content"])
    valid_categories = df["Category"].dropna().unique()

    if result.get("category") is not None:
        if result["category"] not in valid_categories:
            return "Sorry, I could not identify a valid product category."

    valid_channels = ["Online", "Physical Store"]

    if result.get("channel") is not None:
        if result["channel"] not in valid_channels:
            return "Sorry, I could not identify a valid sales channel."

    if result.get("metric", "").lower() != "revenue":
        return "Currently, I can answer revenue questions only."

    revenue = calculate_revenue(
        df,
        year=int(result["year"]) if result.get("year") else None,
        category=result.get("category"),
        channel=result.get("channel")
    )

    return f"Revenue: ${revenue:,.2f}"

In [ ]:
#Now let's test the AI Assistant a complete process using just one line of code.

In [137]:
ask_retail_ai("How much revenue did online computer sales generate in 2019?")

'Revenue: $1,500,897.15'

In [ ]:
#Let's check whether the same function works when the question doesn't mention Online or Physical Store.

In [139]:
ask_retail_ai("What was the total revenue from Computers in 2019?")

'Revenue: $6,958,430.89'

In [141]:
ask_retail_ai("What was the gross profit in 2019?")

'Currently, I can answer revenue questions only.'

In [ ]:
#install Streamlit-to turn our working Python AI Assistant into a simple web application.

In [2]:
%pip show streamlit

Name: streamlit
Version: 1.37.1
Summary: A faster way to build and share data apps
Home-page: https://streamlit.io
Author: Snowflake Inc
Author-email: hello@streamlit.io
License: Apache License 2.0
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: altair, blinker, cachetools, click, gitpython, numpy, packaging, pandas, pillow, protobuf, pyarrow, pydeck, requests, rich, tenacity, toml, tornado, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## Retail Sales AI Assistant — Final Development and Validation

### Objective

Build a simple AI-powered Retail Sales Assistant that understands natural-language business questions and returns accurate revenue calculations from the retail sales dataset.

### Tools Used

- **Python and Pandas:** Data filtering and revenue calculations.
- **Ollama (Llama 3.2):** Interpretation of natural-language business questions.
- **JSON:** Conversion of AI responses into structured information.
- **Streamlit:** Interactive web interface for asking questions and displaying answers.

### How the AI Assistant Works

1. The user enters a business question in Streamlit.
2. Ollama identifies the metric, year, product category, and sales channel.
3. Python validates the extracted information.
4. Pandas filters the dataset and calculates revenue.
5. Streamlit displays the calculated answer.

### Validated Results

| Business Question | Revenue (USD) |
|---|---:|
| Total Computers revenue in 2019 | $6,958,430.89 |
| Online Computers revenue in 2019 | $1,500,897.15 |

### Key Learnings

- Ollama interprets natural-language questions and returns structured JSON.
- Python performs calculations using actual retail sales data rather than relying on AI-generated financial figures.
- Missing filters are handled using Python's `None` value.
- Category and channel validation helps prevent incorrect calculations.
- Streamlit provides a simple interface without requiring users to write Python code.
- Separating AI interpretation from business calculations improves transparency and supports result validation.

### Project Status

**Basic Retail Sales AI Assistant — Completed**

Successfully developed and tested a basic AI-powered application using Ollama, Python, Pandas, and Streamlit.

### Limitations

- The assistant currently supports revenue questions only.
- AI interpretation may not always be accurate for unfamiliar question wording.
- The application runs locally and requires Ollama to be available.

### Conclusion

The project demonstrates how a local large language model can interpret business questions while Python and Pandas perform data-driven calculations. The completed Streamlit application provides a simple, interactive way to explore retail revenue.

### Next Phase

Organize the SQL, Python, Tableau, and AI project files for GitHub documentation and portfolio presentation.